In [1]:
import numpy as np
import json
import cv2


In [18]:
import numpy as np
import json
import cv2

with open("data_cam.json", "r") as f:
    data_cam = json.load(f)
with open("data_cam_fisheye.json", "r") as f:
    data_cam_fisheye = json.load(f)
homography_matrix = np.load("Homography for 229_camera_.npy")

dis_rect_w, dis_rect_h = 2592, 2592
fov = 160.0
fov_RAD = np.deg2rad(fov)
focal = (dis_rect_w / 2) / np.tan(fov_RAD / 2)
dis_K_rect = np.array([[focal, 0.0, dis_rect_w / 2],
                       [0.0, focal, dis_rect_h / 2],
                       [0.0, 0.0, 1.0]])

f_fish = data_cam_fisheye["radius"] / data_cam_fisheye["theta_max_rad"]
K_fish = np.array([[f_fish, 0.0, data_cam_fisheye["cx"]],
                   [0.0, f_fish, data_cam_fisheye["cy"]],
                   [0.0, 0.0, 1.0]])
D_fish = np.zeros((4, 1))

def compute_basis(yaw_deg, pitch_deg):
    yaw_deg_corrected = yaw_deg - 90.0
    yaw, pitch = np.deg2rad([yaw_deg_corrected, pitch_deg])
    Rz = np.array([[np.cos(-yaw), -np.sin(-yaw), 0],
                   [np.sin(-yaw), np.cos(-yaw), 0],
                   [0, 0, 1]])
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(pitch), -np.sin(pitch)],
                   [0, np.sin(pitch), np.cos(pitch)]])
    return np.diag([1, -1, 1]) @ (Rx @ Rz)
rect_w = 1280
rect_h = 720
def rect_to_fisheye_point(u, v, view_params, data_cam_fisheye, rect_w, rect_h):
    Hr, Wr = rect_h, rect_w
    cx_r, cy_r = Wr / 2, Hr / 2
    fov = view_params["fov"]
    f = (Wr / 2) / np.tan(np.deg2rad(fov / 2))
    x = (u - cx_r) / f
    y = (v - cy_r) / f
    z = 1
    ray_rect = np.array([x, y, z])
    ray_rect /= np.linalg.norm(ray_rect)
    Rot = compute_basis(view_params["yaw_deg"], view_params["pitch"])
    ray_fish = Rot.T @ ray_rect
    Xf, Yf, Zf = ray_fish
    theta = np.arccos(np.clip(Zf, -1, 1))
    phi = np.arctan2(Yf, Xf)
    radius, theta_max = data_cam_fisheye["radius"], data_cam_fisheye["theta_max_rad"]
    cx_f, cy_f = data_cam_fisheye["cx"], data_cam_fisheye["cy"]
    r = (theta / theta_max) * radius
    u_f = cx_f + r * np.cos(phi)
    v_f = cy_f - r * np.sin(phi)
    return (float(u_f), float(v_f))
u_considered = 640
v_considered = 360
i = 0
pt_f = rect_to_fisheye_point(u_considered, v_considered, data_cam[i], data_cam_fisheye, rect_w, rect_h)
pt_f_arr = np.array([[[pt_f[0], pt_f[1]]]], dtype=np.float32)
rect_point = cv2.fisheye.undistortPoints(pt_f_arr, K_fish, D_fish, P=dis_K_rect)
mapped_pt = cv2.perspectiveTransform(rect_point, homography_matrix)
print(mapped_pt[0][0][0],mapped_pt[0][0][1])

1289.0839 620.90247


In [ ]:
import json
import numpy as np
import cv2

camera = [
  {"cam_id": "camid_1250", "u": 500, "v": 500},
  {"cam_id": "camid_1251", "u": 125, "v": 345},
  {"cam_id": "camid_1252", "u": 1000, "v": 2000},
  {"cam_id": "camid_1253", "u": 1250, "v": 3450}
]

for i,val in enumerate(camera):
    cam = camera[i]["cam_id"]
    ca , number = cam.split("_")
    cam_number , view = number[:3], number[-1]
    cam_parameters = f"cam_parameters_{cam_number}.json"
    with open(cam_parameters,"r") as f:
        parameter = json.load(f)
    parameter["views"][int(view)]


{'cx': 1296.0, 'cy': 972.0, 'fov': 90.0, 'pitch': 56.0, 'theta_max_deg': 90.0, 'yaw_deg': -18.0}
[[1.0023, 0.0124, -15.32], [-0.0087, 0.9989, 22.41], [1e-05, -2e-05, 1.0]]
{'cx': 1296.0, 'cy': 972.0, 'fov': 90.0, 'pitch': 54.0, 'theta_max_deg': 90.0, 'yaw_deg': -42.0}
[[1.0023, 0.0124, -15.32], [-0.0087, 0.9989, 22.41], [1e-05, -2e-05, 1.0]]
{'cx': 1296.0, 'cy': 972.0, 'fov': 90.0, 'pitch': 28.0, 'theta_max_deg': 90.0, 'yaw_deg': -30.0}
[[1.0023, 0.0124, -15.32], [-0.0087, 0.9989, 22.41], [1e-05, -2e-05, 1.0]]
{'cx': 1296.0, 'cy': 972.0, 'fov': 90.0, 'pitch': 58.0, 'theta_max_deg': 90.0, 'yaw_deg': 24.0}
[[1.0023, 0.0124, -15.32], [-0.0087, 0.9989, 22.41], [1e-05, -2e-05, 1.0]]


In [1]:
camera = [
  {"cam_id": "camid_1250", "u": 500, "v": 500},
  {"cam_id": "camid_1251", "u": 125, "v": 345},
  {"cam_id": "camid_1252", "u": 1000, "v": 2000},
  {"cam_id": "camid_1253", "u": 1250, "v": 3450}
]

In [3]:
import numpy as np
import json
import cv2

def compute_basis(yaw_deg, pitch_deg):
    yaw_deg_corrected = yaw_deg - 90.0
    yaw, pitch = np.deg2rad([yaw_deg_corrected, pitch_deg])
    Rz = np.array([[np.cos(-yaw), -np.sin(-yaw), 0],
                   [np.sin(-yaw), np.cos(-yaw), 0],
                   [0, 0, 1]])
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(pitch), -np.sin(pitch)],
                   [0, np.sin(pitch), np.cos(pitch)]])
    return np.diag([1, -1, 1]) @ (Rx @ Rz)

def rect_to_fisheye_point(u, v, view_params, data_cam_fisheye, rect_w=1280, rect_h=720):
    Hr, Wr = rect_h, rect_w
    cx_r, cy_r = Wr / 2, Hr / 2
    fov = view_params["fov"]
    f = (Wr / 2) / np.tan(np.deg2rad(fov / 2))
    x = (u - cx_r) / f
    y = (v - cy_r) / f
    z = 1
    ray_rect = np.array([x, y, z])
    ray_rect /= np.linalg.norm(ray_rect)
    Rot = compute_basis(view_params["yaw_deg"], view_params["pitch"])
    ray_fish = Rot.T @ ray_rect
    Xf, Yf, Zf = ray_fish
    theta = np.arccos(np.clip(Zf, -1, 1))
    phi = np.arctan2(Yf, Xf)
    radius, theta_max = data_cam_fisheye["radius"], data_cam_fisheye["theta_max_rad"]
    cx_f, cy_f = data_cam_fisheye["cx"], data_cam_fisheye["cy"]
    r = (theta / theta_max) * radius
    u_f = cx_f + r * np.cos(phi)
    v_f = cy_f - r * np.sin(phi)
    return (float(u_f), float(v_f))

def undistort_parameters(data_cam_fisheye):
    dis_rect_w, dis_rect_h = 2592, 2592
    fov = 160.0
    fov_RAD = np.deg2rad(fov)
    focal = (dis_rect_w / 2) / np.tan(fov_RAD / 2)
    dis_K_rect = np.array([[focal, 0.0, dis_rect_w / 2],
                        [0.0, focal, dis_rect_h / 2],
                        [0.0, 0.0, 1.0]])

    f_fish = data_cam_fisheye["radius"] / data_cam_fisheye["theta_max_rad"]
    K_fish = np.array([[f_fish, 0.0, data_cam_fisheye["cx"]],
                    [0.0, f_fish, data_cam_fisheye["cy"]],
                    [0.0, 0.0, 1.0]])
    D_fish = np.zeros((4, 1))
    return dis_K_rect ,K_fish ,D_fish


for i,val in enumerate(camera):
    cam = camera[i]["cam_id"]
    u_point = camera[i]["u"]
    v_point = camera[i]["v"]
    ca , number = cam.split("_")
    cam_number , view = number[:3], number[-1]
    cam_parameters = f"cam_parameters_{cam_number}.json"
    with open(cam_parameters,"r") as f:
        parameter = json.load(f)
    parameter["views"][int(view)]
    H = np.array(parameter["homography"], dtype=np.float64)
    dis_K_rect ,K_fish ,D_fish = undistort_parameters(parameter["fisheye_parameter"])
    
    pt_f = rect_to_fisheye_point(u_point, v_point, parameter["views"][int(view)], parameter["fisheye_parameter"])
    pt_f_arr = np.array([[[pt_f[0], pt_f[1]]]], dtype=np.float32)
    rect_point = cv2.fisheye.undistortPoints(pt_f_arr, K_fish, D_fish, P=dis_K_rect)
    mapped_pt = cv2.perspectiveTransform(rect_point, H)
    print(mapped_pt[0][0][0],mapped_pt[0][0][1])

1545.1586 1321.1016
1776.6581 1292.9421
1113.454 1280.6783
1261.5325 1404.7318
